# 3.1 章节介绍

## 概述

通过前两章学习，相信大家已经掌握了 Vector 算子“读入—计算—写回”的基本计算流程。本章继续以一个简单的 ReLU 算子为示例，和大家一起学习算子开发过程中的 tiling 分核及流水优化相关知识，并介绍 SimdVF 实现与 SimtVF 实现的一些差异。最后，通过 Softmax 算子的实现练习，巩固本章所学知识。

## 学习目标

- 在已有“读入—计算—写回”流程的基础上，理解 tiling 与分核，并能组织大 Tensor 的分块计算和核间任务分配。
- 理解流水优化中搬运与计算的交叠关系，并通过正确性校验和实测判断优化效果。
- 通过相同计算任务的对照，说明 SimdVF 实现与 SimtVF 实现在表达方式和控制粒度上的主要差异。
- 综合运用本章知识完成 Softmax 算子练习，并验证计算结果与性能。

[本章导航](README.md) · [下一节](03.02_single_vector.ipynb)

<img alt="本章学习路线" src="images/03.01_route.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

如上图所示，我们将先实现一个简单vector算子的功能，再学习tiling 分核与流水优化，在此之间也会通过相同计算任务对照 SimdVF 和 SimtVF 的实现方式，最终用 Softmax 实践巩固。图中的流水只表示可能的并行关系，不代表实测情况。

## 本章路线

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">小节</th>
<th style="text-align: left; vertical-align: top;">学习内容</th>
<th style="text-align: left; vertical-align: top;">实践与验证</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="03.02_single_vector.ipynb">3.2</a></td>
<td style="text-align: left; vertical-align: top;">认识 UB空间约束</td>
<td style="text-align: left; vertical-align: top;">运行 ReLU，从实际用例理解Vector 算子对于 UB的限制</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="03.03_tiling.ipynb">3.3</a></td>
<td style="text-align: left; vertical-align: top;">tiling 分核：分块与核间任务分配</td>
<td style="text-align: left; vertical-align: top;">推导每次计算的索引，体会分块与分核的思想</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="03.04_pipeline.ipynb">3.4</a></td>
<td style="text-align: left; vertical-align: top;">流水优化：搬运与计算的并行</td>
<td style="text-align: left; vertical-align: top;">对比流水深度与 buffer 份数，以实测判断效果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="03.05_execution_modes.ipynb">3.5</a></td>
<td style="text-align: left; vertical-align: top;">SimdVF 与 SimtVF 的实现差异</td>
<td style="text-align: left; vertical-align: top;">对照同输入 Add，理解表达方式与控制粒度</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><a href="03.06_softmax.ipynb">3.6</a></td>
<td style="text-align: left; vertical-align: top;">Softmax 综合实践</td>
<td style="text-align: left; vertical-align: top;">实现稳定的行 Softmax，巩固本章学习的内容</td>
</tr>
</tbody>
</table>


## 小结

本章以一个简单的Vector 算子贯穿，3.2 实现 ReLU，从实际用例认识 UB 空间约束；3.3 先将数据分配给多个核，再按各核的 UB 空间进行分块；3.4 学习流水优化，3.5 引入 SimtVF 作对照，3.6 继续使用 SimdVF 完成 Softmax 实践。

## 课后练习

1. 本章将从 ReLU 走向 Softmax。请按学习路线，说明 3.2、3.3、3.4 分别要解决什么问题。
2. 本章为什么安排同一个 Add 的 SimdVF 与 SimtVF 实现对照？学习时应该重点比较哪些方面？

请先结合本章路线回答，再查看解析。

<details>
<summary>展开练习解析</summary>

1. 3.2 完成 ReLU 功能并通过用例认识 UB 容量约束；3.3 研究如何把数据分给多个核，再在核内按 UB 容量分块；3.4 研究多 buffer 如何使搬运与计算交叠。这些知识最终用于 Softmax 实践。

2. 对同一个 Add 使用相同输入和计算任务，可以集中比较两种模式的计算表达、寄存器操作与并行索引方式。GM/UB 搬运与寄存器读写应区分；性能差异需要用相同条件下的 kernel 实测判断。

</details>

[本章导航](README.md) · [下一节](03.02_single_vector.ipynb)